In [4]:
!pip install mysql-connector-python --q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires protobuf<4,>3.12.2, but you have protobuf 4.21.12 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 14.0.1 which is incompatible.
google-cloud-aiplatform 0.6.0a1 requires google-api-core[grpc]<2.0.0dev,>=1.22.2, but you have google-api-core 2.11.1 which is incompatible.
google-cloud-automl 1.0.1 requires google-api-core[grpc]<2.0.0dev,>=1.14.0, but you have google-api-core 2.11.1 which is incompatible.
google-cloud-bigquery 2.34.4 requires protobuf<4.0.0dev,>=3.12.0, but you have protobuf 4.21.12 which is incompatible.
google-cloud-bigtable 1.7.3 requires protobuf<4.0.0dev, but you have protobuf 4.21.12 which is incompatible.
google-cloud-datasto

In [5]:
#mysql installation
!apt-get -y install mysql-server

Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following additional packages will be installed:
  libaio1 libcgi-fast-perl libcgi-pm-perl libencode-locale-perl
  libevent-core-2.1-7 libevent-pthreads-2.1-7 libfcgi-perl libhtml-parser-perl
  libhtml-tagset-perl libhtml-template-perl libhttp-date-perl
  libhttp-message-perl libio-html-perl liblwp-mediatypes-perl libmecab2
  libtimedate-perl liburi-perl mecab-ipadic mecab-ipadic-utf8 mecab-utils
  mysql-client-8.0 mysql-client-core-8.0 mysql-common mysql-server-8.0
  mysql-server-core-8.0 psmisc
Suggested packages:
  libdata-dump-perl libipc-sharedcache-perl libwww-perl mailx tinyca
The following NEW packages will be installed:
  libaio1 libcgi-fast-perl libcgi-pm-perl libencode-locale-perl
  libevent-core-2.1-7 libevent-pthreads-2.1-7 libfcgi-perl libhtml-parser-perl
  libhtml-tagset-perl libhtml-template-perl libhttp-date-perl
  libhttp-message-perl libio-html-perl liblwp-mediatypes-p

In [6]:
# starting the service
!service mysql start

 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.


In [7]:
# creating user id and password for the account
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH 'mysql_native_password' BY 'root';FLUSH PRIVILEGES;"

In [8]:
# creating schema
!mysql -u root -proot -e "CREATE DATABASE classicmodels"

mysql: [Warning] Using a password on the command line interface can be insecure.


In [9]:
# running the db updation file
!mysql -u root -proot classicmodels < /kaggle/input/language-model/mysqlsampledatabase.sql

mysql: [Warning] Using a password on the command line interface can be insecure.


In [10]:
import mysql.connector

# Create a connection to the MySQL server
conn = mysql.connector.connect(user='root', password='root', host='localhost')

# Create a cursor to interact with the MySQL server
cursor = conn.cursor()

In [11]:
cursor.execute("USE classicmodels")

In [12]:
def print_records():
    records = cursor.fetchall()

    # Print the records
    for record in records:
        print(record)

In [13]:
cursor.execute('SHOW TABLES')

In [14]:
print_records()

('customers',)
('employees',)
('offices',)
('orderdetails',)
('orders',)
('payments',)
('productlines',)
('products',)


In [15]:
cursor.execute('select * from customers limit 3')

In [16]:
print_records()

(103, 'Atelier graphique', 'Schmitt', 'Carine ', '40.32.2555', '54, rue Royale', None, 'Nantes', None, '44000', 'France', 1370, Decimal('21000.00'))
(112, 'Signal Gift Stores', 'King', 'Jean', '7025551838', '8489 Strong St.', None, 'Las Vegas', 'NV', '83030', 'USA', 1166, Decimal('71800.00'))
(114, 'Australian Collectors, Co.', 'Ferguson', 'Peter', '03 9520 4555', '636 St Kilda Road', 'Level 3', 'Melbourne', 'Victoria', '3004', 'Australia', 1611, Decimal('117300.00'))


In [17]:
! pip install langchain google-generativeai langchain_experimental pymysql sentence-transformers chromadb --upgrade -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 14.0.1 which is incompatible.
google-cloud-pubsublite 1.8.3 requires overrides<7.0.0,>=6.0.1, but you have overrides 7.4.0 which is incompatible.
jupyterlab 4.0.5 requires jupyter-lsp>=2.0.0, but you have jupyter-lsp 1.5.1 which is incompatible.
jupyterlab-lsp 5.0.0 requires jupyter-lsp>=2.0.0, but you have jupyter-lsp 1.5.1 which is incompatible.
jupyterlab-lsp 5.0.0 requires jupyterlab<5.0.0a0,>=4.0.6, but you have jupyterlab 4.0.5 which is incompatible.
kfp 2.0.1 requires google-cloud-storage<3,>=2.2.1, but you have google-cloud-storage 1.44.0 which is incompatible.
kfp 2.0.1 requires kubernetes<27,>=8.0.0, but you have kubernetes 28.1.0 which 

In [18]:
import langchain
# from langchain.llms import CTransformers
import time

# ### setting PaLM2 LLM model with the help of langchain

from langchain.embeddings import GooglePalmEmbeddings
from langchain.llms import GooglePalm


llm = GooglePalm(google_api_key='XXXXXXX',
                repetition_penalty= 1.2, temperature= 0.0,model_namw='models/code-bison-001',
                top_k=1)

In [20]:
### settting up db connectivity with LangChain

from langchain.utilities import SQLDatabase
from langchain_experimental.sql import SQLDatabaseChain

langchain.verbose = False

db_user = "root"
db_password = "root"
db_host = "localhost"
db_name = "classicmodels"
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=3)

print(db.table_info)


CREATE TABLE customers (
	`customerNumber` INTEGER NOT NULL, 
	`customerName` VARCHAR(50) NOT NULL, 
	`contactLastName` VARCHAR(50) NOT NULL, 
	`contactFirstName` VARCHAR(50) NOT NULL, 
	phone VARCHAR(50) NOT NULL, 
	`addressLine1` VARCHAR(50) NOT NULL, 
	`addressLine2` VARCHAR(50), 
	city VARCHAR(50) NOT NULL, 
	state VARCHAR(50), 
	`postalCode` VARCHAR(15), 
	country VARCHAR(50) NOT NULL, 
	`salesRepEmployeeNumber` INTEGER, 
	`creditLimit` DECIMAL(10, 2), 
	PRIMARY KEY (`customerNumber`), 
	CONSTRAINT customers_ibfk_1 FOREIGN KEY(`salesRepEmployeeNumber`) REFERENCES employees (`employeeNumber`)
)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB

/*
3 rows from customers table:
customerNumber	customerName	contactLastName	contactFirstName	phone	addressLine1	addressLine2	city	state	postalCode	country	salesRepEmployeeNumber	creditLimit
103	Atelier graphique	Schmitt	Carine 	40.32.2555	54, rue Royale	None	Nantes	None	44000	France	1370	21000.00
112	Signal Gift Stores	King	Je

### Creation SQLDBChain

In [21]:
db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True, return_sql=False, use_query_checker=True)

In [23]:
db_chain.run("What are the Order related tables we have in this schema?")



> Entering new SQLDatabaseChain chain...
What are the Order related tables we have in this schema?
SQLQuery:SELECT table_name FROM information_schema.tables WHERE table_schema = 'classicmodels' AND table_name LIKE '%order%'
SQLResult: [('orderdetails',), ('orders',)]
Answer:orderdetails, orders
> Finished chain.


'orderdetails, orders'

In [24]:
db_chain.run("Tell me who are top 5 high performing products in terms of revenue?")



> Entering new SQLDatabaseChain chain...
Tell me who are top 5 high performing products in terms of revenue?
SQLQuery:SELECT productCode, SUM(quantityOrdered * priceEach) AS totalRevenue FROM orderdetails GROUP BY productCode ORDER BY totalRevenue DESC LIMIT 5
SQLResult: [('S18_3232', Decimal('276839.98')), ('S12_1108', Decimal('190755.86')), ('S10_1949', Decimal('190017.96')), ('S10_4698', Decimal('170686.00')), ('S12_1099', Decimal('161531.48'))]
Answer:S18_3232, S12_1108, S10_1949, S10_4698, S12_1099
> Finished chain.


'S18_3232, S12_1108, S10_1949, S10_4698, S12_1099'

In [28]:
db_chain.run("Tell me who are top 5 high performing product names in terms of revenue?")



> Entering new SQLDatabaseChain chain...
Tell me who are top 5 high performing product names in terms of revenue?
SQLQuery:SELECT productName, SUM(quantityOrdered * priceEach) AS total_revenue FROM orderdetails JOIN products ON products.productCode = orderdetails.productCode GROUP BY productName ORDER BY total_revenue DESC LIMIT 5
SQLResult: [('1992 Ferrari 360 Spider red', Decimal('276839.98')), ('2001 Ferrari Enzo', Decimal('190755.86')), ('1952 Alpine Renault 1300', Decimal('190017.96')), ('2003 Harley-Davidson Eagle Drag Bike', Decimal('170686.00')), ('1968 Ford Mustang', Decimal('161531.48'))]
Answer:1992 Ferrari 360 Spider red, 2001 Ferrari Enzo, 1952 Alpine Renault 1300, 2003 Harley-Davidson Eagle Drag Bike, 1968 Ford Mustang
> Finished chain.


'1992 Ferrari 360 Spider red, 2001 Ferrari Enzo, 1952 Alpine Renault 1300, 2003 Harley-Davidson Eagle Drag Bike, 1968 Ford Mustang'

In [30]:
db_chain.run("Tell me who are top 5 customer's name with highest number of orders?")



> Entering new SQLDatabaseChain chain...
Tell me who are top 5 customer's name with highest number of orders?
SQLQuery:SELECT customerName, COUNT(*) AS num_orders FROM customers AS c JOIN orders AS o ON c.customerNumber = o.customerNumber GROUP BY customerName ORDER BY num_orders DESC LIMIT 5
SQLResult: [('Euro+ Shopping Channel', 26), ('Mini Gifts Distributors Ltd.', 17), ('Danish Wholesale Imports', 5), ('Australian Collectors, Co.', 5), ('Dragon Souveniers, Ltd.', 5)]
Answer:Euro+ Shopping Channel, Mini Gifts Distributors Ltd., Danish Wholesale Imports, Australian Collectors, Co., Dragon Souveniers, Ltd.
> Finished chain.


'Euro+ Shopping Channel, Mini Gifts Distributors Ltd., Danish Wholesale Imports, Australian Collectors, Co., Dragon Souveniers, Ltd.'